# CS-4063 <br> Natural Language Processing - Assignment 2

**Muhammad Moiz Khalid** <br>
**23i-2552**<br>
**BDS-6C**

In [ ]:
import re, json, random, time, os
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
os.makedirs("embeddings", exist_ok=True)
import warnings
warnings.filterwarnings("ignore")

## Data Loading

In [ ]:
# load articles from file into {id: text} dict
def loadData(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
    except FileNotFoundError:
        print("Error: Could not find file at {0}".format(file_path))
        return {}

    chunks = re.split(r'\[(\d+)\]', raw_text)

    dataMap = {}

    for i in range(1, len(chunks), 2):
        articleId = int(chunks[i])
        bodyText = chunks[i + 1].strip()

        if bodyText:
            dataMap[articleId] = bodyText

    return dataMap

# strip non-Urdu chars and return word tokens
def tokenize(text):

    cleanText = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    rawWords = cleanText.split()

    tokens = []
    for word in rawWords:

        word = word.strip()

        if word:
            tokens.append(word)

    return tokens

cleaned_articles = loadData('cleaned.txt')
raw_articles= loadData('raw.txt')
metadata = json.load(open('Metadata.json', encoding='utf-8'))

tokenized_cleaned = {aid: tokenize(t) for aid, t in cleaned_articles.items()}
tokenized_raw = {aid: tokenize(t) for aid, t in raw_articles.items()}

doc_ids = sorted(cleaned_articles.keys())

print("Cleaned articles : {}".format(len(cleaned_articles)))
print("Raw articles : {}".format(len(raw_articles)))
print("Metadata entries : {}".format(len(metadata)))

### Topic Assignment

In [ ]:
topicCategories = {
    'Politics': ['الیکشن', 'حکومت', 'وزیر', 'پارلیمنٹ', 'سیاست','جماعت', 'ووٹ', 'وزیراعظم', 'صدر', 'اسمبلی'],
    'Sports': ['کرکٹ', 'میچ', 'ٹیم', 'کھلاڑی', 'ورلڈکپ','فٹبال', 'کھیل', 'ٹورنامنٹ', 'وکٹ', 'گول'],
    'Economy': ['مہنگائی', 'تجارت', 'بینک', 'بجٹ', 'معیشت','روپیہ', 'قرض', 'سرمایہ', 'ٹیکس', 'ڈالر'],
    'International': ['اقوام', 'معاہدہ', 'امریکہ', 'چین', 'ایران','یوکرین', 'اسرائیل', 'روس', 'غیرملکی', 'تنازع'],
    'Health_Society': ['ہسپتال', 'بیماری', 'ویکسین', 'سیلاب', 'تعلیم','صحت', 'وبا', 'ڈاکٹر', 'آفت', 'امداد'],}

# score each category by keyword hits, pick best
def assign_topic(articleId):
    title = metadata.get(str(articleId), {}).get('title', '')
    body = cleaned_articles.get(articleId, '')
    combined = "{}{}".format(title, body)

    scores = {}

    for topic, keywords in topicCategories.items():
        totalScore = 0
        for word in keywords:
            totalScore += combined.count(word)
        scores[topic] = totalScore

    bestTopic = max(scores, key=scores.get)

    if scores[bestTopic] > 0:
        return bestTopic
    else:
        return 'International'

articleTopics = {}

for aid in cleaned_articles:
    topic = assign_topic(aid)
    articleTopics[aid] = topic

print("Topic distribution:")

topicCounts = Counter(articleTopics.values())
sortedItems = sorted(topicCounts.items())

for topic, count in sortedItems:
    print("{}: {}".format(topic, count))

## 1.1 TF-IDF Weighting

In [ ]:
# build vocab from top 10k tokens
maxVocab = 10_000

allTokens = []
for tokens in tokenized_cleaned.values():
    for tok in tokens:
        allTokens.append(tok)

tokenFreq = Counter(allTokens)

topVocabItems = tokenFreq.most_common(maxVocab)
topVocab = []
for word, count in topVocabItems:
    topVocab.append(word)

vocab = {}
fullList = ['<UNK>'] + topVocab
for i, w in enumerate(fullList):
    vocab[w] = i

invVocab = {}
for w, i in vocab.items():
    invVocab[i] = w

V = len(vocab)
D = len(cleaned_articles)

print("Vocabulary size (incl. <UNK>): {}".format(V))
print("Number of documents: {}".format(D))

In [ ]:
# build term-document matrix then compute TF-IDF
wordDocCounts = np.zeros((V, D), dtype=np.float32)
for dIdx, aid in enumerate(doc_ids):
    for tok in tokenized_cleaned[aid]:
        wordDocCounts[vocab.get(tok, 0), dIdx] += 1

docLengths = wordDocCounts.sum(axis=0, keepdims=True)
docLengths[docLengths == 0] = 1
TF = wordDocCounts / docLengths

DF = (wordDocCounts > 0).sum(axis=1)
IDF = np.log(D / (1 + DF))

tfidfMatrix = TF * IDF[:, np.newaxis]

np.save('embeddings/tfidf_matrix.npy', tfidfMatrix)
print("TF-IDF matrix shape : {0}".format(tfidfMatrix.shape))
print("Saved embeddings/tfidf_matrix.npy")

In [ ]:
print("Top-10 most discriminative words per topic (mean TF-IDF):\n")
for topic in topicCategories:
    topicDocIndices = [i for i, aid in enumerate(doc_ids) if articleTopics[aid] == topic]

    if not topicDocIndices:
        continue

    meanTfidf = tfidfMatrix[:, topicDocIndices].mean(axis=1)
    top10Ids = np.argsort(meanTfidf)[::-1][:15]
    top10Words = [invVocab[i] for i in top10Ids if invVocab[i] != '<UNK>'][:10]

    print("  {0}:".format(topic))
    for rank, w in enumerate(top10Words, 1):
        print("    {0:2d}. {1}".format(rank, w))
    print()

## 1.2 Pointwise Mutual Information (PPMI)

In [ ]:
# build co-occurrence matrix with window k=5
windowK = 5
PPMIvocabN = 2000

ppmiVocabWords = topVocab[:PPMIvocabN - 1]
ppmiVocab  = {w: i + 1 for i, w in enumerate(ppmiVocabWords)}
ppmiVocab['<UNK>'] = 0
ppmiV = len(ppmiVocab)
invPpmiVocab = {i: w for w, i in ppmiVocab.items()}

print("PPMI vocab size : {0}".format(ppmiV))
print("Context window : k = {0}".format(windowK))
print("Building co-occurrence matrix ...")

cooc = np.zeros((ppmiV, ppmiV), dtype=np.float32)
for aid in doc_ids:
    ids = [ppmiVocab.get(t, 0) for t in tokenized_cleaned[aid]]
    for i, center in enumerate(ids):
        if center == 0:
            continue
        left  = max(0, i - windowK)
        right = min(len(ids), i + windowK + 1)
        for j in range(left, right):
            if j != i and ids[j] != 0:
                cooc[center, ids[j]] += 1

rowSums = cooc.sum(axis=1, keepdims=True)
colSums = cooc.sum(axis=0, keepdims=True)
total = cooc.sum()
rowSums[rowSums == 0] = 1
colSums[colSums == 0] = 1

# compute PMI then clip negatives to get PPMI
with np.errstate(divide='ignore', invalid='ignore'):
    pmi = np.log2((cooc * total) / (rowSums * colSums))
    pmi[~np.isfinite(pmi)] = 0

ppmiMatrix = np.maximum(pmi, 0)

np.save('embeddings/ppmi_matrix.npy', ppmiMatrix)
print("PPMI matrix shape : {0}".format(ppmiMatrix.shape))
print("Saved embeddings/ppmi_matrix.npy")

In [ ]:
print("Running t-SNE")

tsneWords = [w for w in ppmiVocabWords if w in ppmiVocab][:200]
tsneIds = [ppmiVocab[w] for w in tsneWords]
vecs200 = ppmiMatrix[tsneIds, :]

svd     = TruncatedSVD(n_components=50, random_state=42)
reduced = svd.fit_transform(vecs200)

tsne   = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
coords = tsne.fit_transform(reduced)

wordTopicMap = {}
for aid, tokens in tokenized_cleaned.items():
    tp = articleTopics[aid]
    for tok in tokens:
        if tok not in wordTopicMap:
            wordTopicMap[tok] = Counter()
        wordTopicMap[tok][tp] += 1

TOPIC_COLORS = {'Politics': 'red', 'Sports': 'blue', 'Economy': 'green','International': 'orange', 'Health_Society': 'purple'}

def word_color(w):
    if w not in wordTopicMap:
        return 'gray'
    best = wordTopicMap[w].most_common(1)[0][0]
    return TOPIC_COLORS.get(best, 'gray')

colors = [word_color(w) for w in tsneWords]

fig, ax = plt.subplots(figsize=(14, 10))
for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, c=colors[i], s=18, alpha=0.7)
    if i % 10 == 0:
        ax.annotate(tsneWords[i], (x, y), fontsize=5, alpha=0.65)

legendElements = [Patch(facecolor=v, label=k) for k, v in TOPIC_COLORS.items()] + [Patch(facecolor='gray', label='Other')]
ax.legend(handles=legendElements, loc='upper right', title='Topic')
ax.set_title('t-SNE of 200 Most Frequent Tokens (PPMI vectors, colour-coded by topic)')
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
plt.tight_layout()
plt.savefig('embeddings/tsne_ppmi.png', dpi=150)
plt.show()
print("Saved embeddings/tsne_ppmi.png")

In [ ]:
ppmiNorms  = np.linalg.norm(ppmiMatrix, axis=1, keepdims=True)
ppmiNorms[ppmiNorms == 0] = 1
ppmiNormed = ppmiMatrix / ppmiNorms

queryWordsPpmi = ['پاکستان', 'کرکٹ', 'حکومت', 'معیشت', 'فوج', 'صحت', 'تعلیم', 'بینک', 'عدالت', 'سیاست']

print("Top-5 nearest neighbours in PPMI space:\n")
for qw in queryWordsPpmi:
    qid = ppmiVocab.get(qw)
    if qid is None or qid == 0:
        print("  '{0}': not in PPMI vocab".format(qw))
        continue
    sims = ppmiNormed @ ppmiNormed[qid]
    top5 = [invPpmiVocab[i] for i in np.argsort(sims)[::-1][1:6]]
    print("  '{0}': {1}".format(qw, top5))

## 2.1 Skip-gram Word2Vec

### Vocabulary & Noise Distribution

In [ ]:
freqArray = np.zeros(V, dtype=np.float32)
for w, cnt in tokenFreq.items():
    if w in vocab:
        freqArray[vocab[w]] = cnt
freqArray[0] = 0

noiseDist = freqArray ** 0.75
noiseDist = noiseDist / noiseDist.sum()

print("Vocabulary size : {0}".format(V))
print("Noise table sum : {0:.6f}".format(noiseDist.sum()))
print("\nTop-10 most frequent words:")
for w, c in tokenFreq.most_common(10):
    print("  {0:<20} freq={1:6d}  p_noise={2:.6f}".format(w, c, noiseDist[vocab.get(w,0)]))

### Skip-gram Dataset

In [ ]:
D_EMBED    = 100
W2V_WINDOW = 5
K_NOISE    = 10
BATCH_SIZE = 512
EPOCHS     = 5
LR         = 0.001

# dataset yields (center, context, negatives) triplets
class SkipGramDataset(Dataset):
    """Generates (center, context, K_noise_negatives) triplets."""
    def __init__(self, articles, word2idx, window=5, K=10):
        self.pairs = []
        self.K     = K
        for tokens in articles.values():
            ids = [word2idx.get(t, 0) for t in tokens]
            for i, center in enumerate(ids):
                left  = max(0, i - window)
                right = min(len(ids), i + window + 1)
                for j in range(left, right):
                    if j != i:
                        self.pairs.append((center, ids[j]))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        center, context = self.pairs[idx]
        negatives = np.random.choice(len(noiseDist), size=self.K, p=noiseDist)
        return (torch.tensor(center,    dtype=torch.long),
                torch.tensor(context,   dtype=torch.long),
                torch.tensor(negatives, dtype=torch.long))

datasetClean = SkipGramDataset(tokenized_cleaned, vocab, W2V_WINDOW, K_NOISE)
datasetRaw   = SkipGramDataset(tokenized_raw,     vocab, W2V_WINDOW, K_NOISE)

print("Skip-gram pairs (cleaned) : {0:,}".format(len(datasetClean)))
print("Skip-gram pairs (raw) : {0:,}".format(len(datasetRaw)))

### Skip-gram Model Definition

In [ ]:
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.V = nn.Embedding(vocab_size, embed_dim)
        self.U = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.V.weight, -0.5 / embed_dim, 0.5 / embed_dim)
        nn.init.zeros_(self.U.weight)

    def forward(self, centers, contexts, negatives):
        vc = self.V(centers)
        uo = self.U(contexts)
        un = self.U(negatives)

        pos_score = torch.sum(vc * uo, dim=1)
        neg_score = torch.bmm(un, vc.unsqueeze(2)).squeeze(2)

        loss = -torch.mean(
            torch.log(torch.sigmoid(pos_score) + 1e-10) +
            torch.sum(torch.log(torch.sigmoid(-neg_score) + 1e-10), dim=1)
        )
        return loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device : {0}".format(device))


### Training Loop

In [ ]:
# train one skip-gram condition and return embeddings + loss history
def train_skipgram(dataset, vocab_size, embed_dim, epochs=5,batch_size=512, lr=0.001, label=''):
    """Train Skip-gram model; return averaged embeddings and loss history."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    model = SkipGram(vocab_size, embed_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    intervalLosses, intervalSteps = [], []
    reportEvery = max(1, len(loader) // 10)
    globalStep = 0

    print("Training: {0}  |  dim={1}  |  pairs={2:,}".format(label, embed_dim, len(dataset)))
    print("Epochs={0}  BatchSize={1}  LR={2}".format(epochs, batch_size, lr))

    for epoch in range(epochs):
        model.train()
        runningLoss = 0.0
        t0 = time.time()

        for step, (centers, contexts, negatives) in enumerate(loader):
            centers   = centers.to(device)
            contexts  = contexts.to(device)
            negatives = negatives.to(device)

            optimizer.zero_grad()
            loss = model(centers, contexts, negatives)
            loss.backward()
            optimizer.step()

            runningLoss += loss.item()
            globalStep  += 1

            if (step + 1) % reportEvery == 0:
                avg = runningLoss / reportEvery
                intervalLosses.append(avg)
                intervalSteps.append(globalStep)
                print("  Epoch {0}/{1} | Step {2:>5}/{3} | Loss: {4:.4f}".format(epoch+1, epochs, step+1, len(loader), avg))
                runningLoss = 0.0

        print("  Epoch {0} done in {1:.1f}s".format(epoch+1, time.time()-t0))

    with torch.no_grad():
        avgEmb = 0.5 * (model.V.weight.cpu().numpy() + model.U.weight.cpu().numpy())

    return model, avgEmb, intervalLosses, intervalSteps

### C3) Skip-gram on `cleaned.txt`, d=100

In [ ]:
modelC3, embC3, lossesC3, stepsC3 = train_skipgram(datasetClean, V, D_EMBED, EPOCHS, BATCH_SIZE, LR,label='C3: cleaned.txt  d=100')

np.save('embeddings/embeddings_w2v.npy', embC3)
np.save('embeddings/embC3.npy', embC3)
print("\nSaved embeddings/embeddings_w2v.npy")
print("Saved embeddings/embC3.npy")

plt.figure(figsize=(8, 4))
plt.plot(stepsC3, lossesC3, label='C3 (cleaned, d=100)')
plt.xlabel('Training Step')
plt.ylabel('BCE Loss')
plt.title('Skip-gram Training Loss — C3 (cleaned.txt, d=100)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('embeddings/loss_curve_c3.png', dpi=150)
plt.show()
print("Saved embeddings/loss_curve_c3.png")

### C2) Skip-gram on `raw.txt`, d=100

In [ ]:
modelC2, embC2, lossesC2, stepsC2 = train_skipgram(datasetRaw, V, D_EMBED, EPOCHS, BATCH_SIZE, LR,label='C2: raw.txt  d=100')

np.save('embeddings/embC2.npy', embC2)
print("Saved embeddings/embC2.npy")

### C4) Skip-gram on `cleaned.txt`, d=200

In [ ]:
modelC4, embC4, lossesC4, stepsC4 = train_skipgram(datasetClean, V, 200, EPOCHS, BATCH_SIZE, LR,label='C4: cleaned.txt  d=200')

np.save('embeddings/embC4.npy', embC4)
print("Saved embeddings/embC4.npy")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stepsC4, lossesC4, label='C4: cleaned d=200', linestyle=':')
ax.set_xlabel('Training Step'); ax.set_ylabel('BCE Loss')
ax.set_title('Skip-gram Training Loss — Conditions C2, C3, C4')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('embeddings/loss_curve_c4.png', dpi=150)
plt.show()
print("Saved embeddings/loss_curve_c4.png")

In [ ]:
json.dump(vocab, open('embeddings/word2idx.json', 'w', encoding='utf-8'),ensure_ascii=False)
print("Saved embeddings/word2idx.json")

## 2.2 Evaluation

In [ ]:
# return top-n nearest neighbours by cosine similarity
def get_top_n(emb, word2idx, invVocab, query_word, n=10):
    """Top-n nearest neighbours by cosine similarity."""
    if query_word not in word2idx:
        return f"'{query_word}' not in vocabulary"
    qid  = word2idx[query_word]
    qvec = emb[qid]
    nrms = np.linalg.norm(emb, axis=1) + 1e-10
    sims = (emb @ qvec) / (np.linalg.norm(qvec) * nrms)
    ids  = np.argsort(sims)[::-1][1:n + 1]
    return [(invVocab[i], round(float(sims[i]), 4)) for i in ids]

# solve v(b) - v(a) + v(c) analogy
def vector_analogy(emb, word2idx, invVocab, a, b, c, n=3):
    """v(b) - v(a) + v(c); return top-n candidates."""
    for w in [a, b, c]:
        if w not in word2idx:
            return f"'{w}' not in vocabulary", []
    target = emb[word2idx[b]] - emb[word2idx[a]] + emb[word2idx[c]]
    target /= (np.linalg.norm(target) + 1e-10)
    nrms    = np.linalg.norm(emb, axis=1, keepdims=True); nrms[nrms == 0] = 1
    normed  = emb / nrms
    sims    = normed @ target
    exclude = {word2idx[a], word2idx[b], word2idx[c]}
    ids     = [i for i in np.argsort(sims)[::-1] if i not in exclude][:n]
    return None, [(invVocab[i], round(float(sims[i]), 4)) for i in ids]

print("Evaluation helpers defined.")

### Top-10 Nearest Neighbours for Required Query Words

In [ ]:
QUERY_URDU = {
    'Pakistan' : 'پاکستان',
    'Hukumat'  : 'حکومت',
    'Adalat'   : 'عدالت',
    'Maeeshat' : 'معیشت',
    'Fauj'     : 'فوج',
    'Sehat'    : 'صحت',
    'Taleem'   : 'تعلیم',
    'Aabadi'   : 'آبادی',
}
folder = "embeddings"
embC3 = np.load(os.path.join(folder, "embC3.npy"))

print("Top-10 Nearest Neighbours  C3 embeddings\n")
for eng, urdu in QUERY_URDU.items():
    result = get_top_n(embC3, vocab, invVocab, urdu, n=10)
    print("  {0} ({1}):".format(eng, urdu))
    if isinstance(result, str):
        print("    {0}".format(result))
    else:
        for rank, (w, s) in enumerate(result, 1):
            print("    {0:2d}. {1:<25} ({2:.4f})".format(rank, w, s))
    print()

### Analogy Tests — v(b) − v(a) + v(c)

In [ ]:
ANALOGY_TESTS = [
    ('وزیر',   'حکومت',  'جج',      'عدالت'),
    ('فوج',    'افواج',  'کرکٹ',    'بورڈ'),
    ('کرکٹ',   'میچ',    'فوج',     'مسلح'),
    ('وزیر',   'حکومت',  'مقدمے',   'عدالت'),
    ('کرکٹ',   'ٹیم',    'فوج',     'مسلح'),
    ('صحت',    'ذہنی',   'معیشت',   'صنعت'),
    ('تعلیم',  'مدرسے',  'صحت',     'ذہنی'),
    ('پاکستان','انڈیا',  'فوج',     'مسلح'),
    ('معیشت',  'صنعت',   'صحت',     'مند'),
    ('تعلیم',  'یافتہ',  'صحت',     'مند'),
]

print("Analogy Tests (C3 embeddings)\n")
correct = 0
for a, b, c, expected in ANALOGY_TESTS:
    err, results = vector_analogy(embC3, vocab, invVocab, a, b, c, n=3)
    if err:
        print("   {} : {} :: {} : ?  -> ERROR: {}".format(a, b, c, err)); continue
    top3 = [r[0] for r in results]
    ok   = any(expected in w or w in expected for w in top3)
    if ok: correct += 1
    print("  {0}  {1} : {2} :: {3} : ?".format(' ' if ok else ' ', a, b, c))
    print("      Top-3 : {0}".format(top3))
    print("      Expected concept : '{0}'\n".format(expected))

print("Correct analogies : {0} / 10".format(correct))

## Condition Comparison (C1 – C4)

In [ ]:
# mean reciprocal rank over labelled word pairs
def mrr_skipgram(emb, word2idx, pairs):
    """Mean Reciprocal Rank over labelled word pairs."""
    rrs = []
    for query, gold in pairs:
        if query not in word2idx or gold not in word2idx:
            rrs.append(0.0); continue
        qvec  = emb[word2idx[query]]
        nrms  = np.linalg.norm(emb, axis=1) + 1e-10
        sims  = (emb @ qvec) / (np.linalg.norm(qvec) * nrms)
        ranked = np.argsort(sims)[::-1]
        rank  = np.where(ranked == word2idx[gold])[0]
        rrs.append(1.0 / (rank[0] + 1) if len(rank) > 0 else 0.0)
    return float(np.mean(rrs))

def mrr_ppmi(pairs):
    """MRR using PPMI cosine similarity."""
    ppmiMatrix = np.load(os.path.join(folder, "ppmiMatrix.npy"))
    rrs = []
    ppmiNorms = np.linalg.norm(ppmiMatrix, axis=1) + 1e-10
    for query, gold in pairs:
        if query not in ppmiVocab or gold not in ppmiVocab:
            rrs.append(0.0); continue
        qid   = ppmiVocab[query]; gid = ppmiVocab[gold]
        qvec  = ppmiMatrix[qid]
        sims  = (ppmiMatrix @ qvec) / (np.linalg.norm(qvec) * ppmiNorms)
        ranked = np.argsort(sims)[::-1]
        rank  = np.where(ranked == gid)[0]
        rrs.append(1.0 / (rank[0] + 1) if len(rank) > 0 else 0.0)
    return float(np.mean(rrs))

LABELLED_PAIRS = [
    ('کرکٹ',   'میچ'),
    ('کرکٹ',   'ٹیم'),
    ('کرکٹ',   'بورڈ'),
    ('فوج',    'مسلح'),
    ('فوج',    'فوجی'),
    ('فوج',    'افواج'),
    ('عدالت',  'مقدمے'),
    ('عدالت',  'کورٹ'),
    ('عدالت',  'جسٹس'),
    ('حکومت',  'وزیر'),
    ('حکومت',  'صوبائی'),
    ('پاکستان','انڈیا'),
    ('پاکستان','دنیا'),
    ('صحت',    'ذہنی'),
    ('صحت',    'مند'),
    ('تعلیم',  'مدرسے'),
    ('تعلیم',  'یافتہ'),
    ('معیشت',  'صنعت'),
    ('معیشت',  'فروغ'),
    ('ایران',  'ایرانی'),
]

folder = "embeddings"

embC2 = np.load(os.path.join(folder, "embC2.npy"))
embC3 = np.load(os.path.join(folder, "embC3.npy"))
embC4 = np.load(os.path.join(folder, "embC4.npy"))

mrr_c1 = mrr_ppmi(LABELLED_PAIRS)
mrr_c2 = mrr_skipgram(embC2, vocab, LABELLED_PAIRS)
mrr_c3 = mrr_skipgram(embC3, vocab, LABELLED_PAIRS)
mrr_c4 = mrr_skipgram(embC4, vocab, LABELLED_PAIRS)

print("\nFour-Condition Comparison — MRR on 20 labelled pairs")
print( )
print("{0:<5} {1:<35} {2:>8}".format('ID', 'Condition', 'MRR'))
print()
print("{0:<5} {1:<35} {2:>8.4f}".format('C1', 'PPMI baseline', mrr_c1))
print("{0:<5} {1:<35} {2:>8.4f}".format('C2', 'Skip-gram on raw.txt (d=100)', mrr_c2))
print("{0:<5} {1:<35} {2:>8.4f}".format('C3', 'Skip-gram on cleaned.txt (d=100)', mrr_c3))
print("{0:<5} {1:<35} {2:>8.4f}".format('C4', 'Skip-gram on cleaned.txt (d=200)', mrr_c4))

In [ ]:
EVAL_WORDS = ['پاکستان', 'حکومت', 'کرکٹ', 'فوج', 'عدالت']

print("Top-5 nearest neighbours per condition\n")
for qw in EVAL_WORDS:
    print("  Query: '{0}'".format(qw))

    if qw in ppmiVocab:
        qid  = ppmiVocab[qw]
        nrms = np.linalg.norm(ppmiMatrix, axis=1) + 1e-10
        sims = (ppmiMatrix @ ppmiMatrix[qid]) / (np.linalg.norm(ppmiMatrix[qid]) * nrms)
        top5 = [invPpmiVocab[i] for i in np.argsort(sims)[::-1][1:6]]
        print("    C1 (PPMI)      : {0}".format(top5))
    else:
        print("    C1 (PPMI)      : not in vocab")

    for lbl, emb in [('C2 (raw d=100)', embC2),
                     ('C3 (clean d=100)', embC3),
                     ('C4 (clean d=200)', embC4)]:
        r = get_top_n(emb, vocab, invVocab, qw, n=5)
        words = [w for w, _ in r] if isinstance(r, list) else r
        print("    {0:<20}: {1}".format(lbl, words))
    print()

## Part 2) Sequence Labeling: POS Tagging & NER

In [ ]:
folder = "embeddings"
embC2 = np.load(os.path.join(folder, "embC2.npy"))
embC3 = np.load(os.path.join(folder, "embC3.npy"))
embC4 = np.load(os.path.join(folder, "embC4.npy"))

## 3. Dataset Preparation

In [ ]:
import os, re, json, random
from collections import Counter, defaultdict
import numpy as np

random.seed(42)
np.random.seed(42)

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# split Urdu text on sentence-final punctuation
def split_sentences(text):
    """Split Urdu text into sentences on ۔ or newline."""
    sents = re.split(r'[۔\n]+', text)
    return [s.strip() for s in sents if len(s.strip().split()) >= 3]

topicSentences = defaultdict(list)
for aid, tokens_list in tokenized_cleaned.items():
    topic = articleTopics[aid]
    text  = cleaned_articles[aid]
    for sent in split_sentences(text):
        toks = tokenize(sent)
        if len(toks) >= 3:
            topicSentences[topic].append((aid, toks))

print("Sentences per topic (before sampling):")
for t, sents in topicSentences.items():
    print("  {0:<20}: {1}".format(t, len(sents)))

TOTAL_SENTS = 500
MIN_PER_TOPIC = 100

sortedTopics = sorted(topicSentences.keys(), key=lambda t: len(topicSentences[t]), reverse=True)
selected = []

for t in sortedTopics[:3]:
    pool = topicSentences[t]
    take = random.sample(pool, min(MIN_PER_TOPIC, len(pool)))
    for aid, toks in take:
        selected.append({'aid': aid, 'topic': t, 'tokens': toks})

used = set(id(x) for x in selected)
remainingPool = []
for t, sents in topicSentences.items():
    for item in sents:
        obj = {'aid': item[0], 'topic': t, 'tokens': item[1]}
        remainingPool.append(obj)

random.shuffle(remainingPool)
for obj in remainingPool:
    if len(selected) >= TOTAL_SENTS:
        break
    if obj['tokens'] not in [s['tokens'] for s in selected]:
        selected.append(obj)

random.shuffle(selected)
print("\nTotal selected sentences : {0}".format(len(selected)))
print("Topic distribution in selected set:")
for t, cnt in Counter(s['topic'] for s in selected).items():
    print("  {0:<20}: {1}".format(t, cnt))

### 3.2 POS Annotation (Rule-Based Tagger + Lexicon)

In [ ]:
POS_LEXICON = {}

NOUNS = [
    'آدمی','عورت','بچہ','بچی','گھر','کمرہ','دروازہ','کھڑکی','میز','کرسی',
    'کتاب','قلم','کاغذ','پانی','کھانا','روٹی','چاول','دودھ','چائے','کپ',
    'دن','رات','صبح','شام','وقت','سال','مہینہ','ہفتہ','گھنٹہ','لمحہ',
    'شہر','گاؤں','ملک','دنیا','زمین','آسمان','پہاڑ','دریا','سمندر','جنگل',
    'حکومت','وزیر','صدر','وزیراعظم','اسمبلی','پارلیمنٹ','الیکشن','ووٹ','پارٹی','سیاست',
    'عدالت','جج','مقدمہ','فیصلہ','قانون','پولیس','جیل','ملزم','وکیل','گواہ',
    'فوج','سپاہی','جنرل','جنگ','امن','دشمن','سرحد','حملہ','دفاع','اسلحہ',
    'اسکول','یونیورسٹی','استاد','طالبعلم','امتحان','کلاس','کتب خانہ','سرٹیفکیٹ','ڈگری','تعلیم',
    'ہسپتال','ڈاکٹر','مریض','دوائی','علاج','بیماری','آپریشن','نرس','صحت','وبا',
    'بینک','روپیہ','ڈالر','بجٹ','تجارت','معیشت','قرض','سرمایہ','مہنگائی','ٹیکس',
    'کرکٹ','میچ','ٹیم','کھلاڑی','وکٹ','رنز','بیٹ','گیند','اسٹیڈیم','ٹورنامنٹ',
    'پاکستان','انڈیا','بھارت','چین','امریکہ','ایران','روس','برطانیہ','فرانس','جاپان',
    'لاہور','کراچی','اسلام آباد','پشاور','کوئٹہ','ملتان','فیصل آباد','راولپنڈی','حیدرآباد','سکھر',
    'پنجاب','سندھ','بلوچستان','خیبر','کشمیر','گلگت','آزاد','وفاق','صوبہ','ضلع',
    'اخبار','خبر','رپورٹ','بیان','اعلان','پریس','میڈیا','ٹی وی','ریڈیو','انٹرنیٹ',
    'مسجد','چرچ','مندر','گرودوارہ','مذہب','اسلام','نماز','روزہ','حج','عید',
    'باپ','ماں','بھائی','بہن','بیٹا','بیٹی','دادا','دادی','چچا','پھوپھی',
    'دوست','دشمن','پڑوسی','رشتہ دار','ساتھی','حریف','مہمان','میزبان','سربراہ','نمائندہ',
    'سڑک','پل','ریلوے','جہاز','کار','بس','ٹرین','بندرگاہ','ہوائی اڈہ','اسٹیشن',
    'بارش','سیلاب','طوفان','زلزلہ','آگ','دھواں','گرمی','سردی','خشک سالی','موسم',
]
for w in NOUNS:
    POS_LEXICON[w] = 'NOUN'

VERBS = [
    'ہے','ہیں','تھا','تھی','تھے','ہو','ہوں','ہوتا','ہوتی','ہوتے',
    'کیا','کی','کیے','کرنا','کرتا','کرتی','کرتے','کریں','کرے','کرو',
    'گیا','گئی','گئے','جانا','جاتا','جاتی','جاتے','جائیں','جائے','چلا',
    'آیا','آئی','آئے','آنا','آتا','آتی','آتے','آئیں','آجائے','آجانا',
    'کہا','کہی','کہے','کہنا','کہتا','کہتی','کہتے','کہیں','بولا','بولنا',
    'دیا','دی','دیے','دینا','دیتا','دیتی','دیتے','دیں','ملا','ملنا',
    'لیا','لی','لیے','لینا','لیتا','لیتی','لیتے','لیں','پایا','پانا',
    'بنا','بنی','بنے','بنانا','بناتا','بناتی','بناتے','بنیں','بنائی','بنائے',
    'دیکھا','دیکھی','دیکھے','دیکھنا','دیکھتا','دیکھتی','دیکھتے','دیکھیں','دکھایا','دکھانا',
    'سنا','سنی','سنے','سننا','سنتا','سنتی','سنتے','سنیں','پڑھا','پڑھنا',
    'لکھا','لکھی','لکھے','لکھنا','لکھتا','لکھتی','لکھتے','لکھیں','سمجھا','سمجھنا',
    'چاہا','چاہی','چاہیے','چاہنا','چاہتا','چاہتی','چاہتے','چاہیں','مانا','ماننا',
    'ہوا','ہوئی','ہوئے','ہونا','ہوتا','ہوتی','ہوتے','ہوجانا','ہوسکتا','ہوسکتی',
    'رہا','رہی','رہے','رہنا','رہتا','رہتی','رہتے','رہیں','ٹھہرا','ٹھہرنا',
    'کھایا','کھائی','کھائے','کھانا','پیا','پینا','سویا','سونا','اٹھا','اٹھنا',
    'مارا','ماری','ماریں','مارنا','توڑا','توڑنا','جلایا','جلانا','بھیجا','بھیجنا',
    'نکلا','نکلی','نکلے','نکلنا','پہنچا','پہنچنا','اترا','اترنا','چڑھا','چڑھنا',
    'روکا','روکنا','چھوڑا','چھوڑنا','ملایا','ملانا','بڑھایا','بڑھانا','گھٹایا','گھٹانا',
    'شروع','ختم','جاری','بند','کھلا','بند','چالو','بحال','معطل','منسوخ',
    'اعلان','بیان','تصدیق','تردید','انکشاف','اعتراف','دعوی','الزام','مطالبہ','درخواست',
]
for w in VERBS:
    POS_LEXICON[w] = 'VERB'

ADJS = [
    'بڑا','بڑی','بڑے','چھوٹا','چھوٹی','چھوٹے','اچھا','اچھی','اچھے','برا',
    'نیا','نئی','نئے','پرانا','پرانی','پرانے','لمبا','لمبی','لمبے','چھوٹا',
    'سفید','کالا','سرخ','سبز','نیلا','پیلا','گلابی','بھورا','نارنجی','جامنی',
    'گرم','ٹھنڈا','خشک','گیلا','سخت','نرم','بھاری','ہلکا','تیز','سست',
    'خوبصورت','بدصورت','صاف','گندا','صحیح','غلط','سچا','جھوٹا','ایماندار','بے ایمان',
    'امیر','غریب','طاقتور','کمزور','ذہین','بیوقوف','جوان','بوڑھا','تندرست','بیمار',
    'خوش','ناخوش','خوفزدہ','بہادر','ڈرپوک','شریف','بدمعاش','محنتی','سست','مشہور',
    'ملکی','غیرملکی','قومی','بین الاقوامی','مقامی','علاقائی','وفاقی','صوبائی','ضلعی','شہری',
    'سیاسی','اقتصادی','سماجی','مذہبی','فوجی','عدالتی','تعلیمی','صحت','ثقافتی','میڈیا',
    'پہلا','دوسرا','تیسرا','چوتھا','پانچواں','آخری','اگلا','پچھلا','موجودہ','سابق',
    'نئی','پرانی','تازہ','قدیم','جدید','روایتی','عصری','مستقبل','ماضی','حال',
    'سالانہ','ماہانہ','ہفتہ وار','روزانہ','عارضی','مستقل','وقتی','دیرپا','فوری','تاخیری',
]
for w in ADJS:
    POS_LEXICON[w] = 'ADJ'

PRONS = [
    'میں','ہم','تم','آپ','وہ','یہ','اس','ان','اسے','انہیں',
    'مجھے','ہمیں','تمہیں','آپ کو','اس کو','ان کو','جو','کون','کیا','کوئی',
    'کچھ','سب','ہر','بعض','دونوں','خود','اپنا','اپنی','اپنے','جس',
]
for w in PRONS:
    POS_LEXICON[w] = 'PRON'

DETS = [
    'یہ','وہ','اس','ان','کوئی','کچھ','ہر','سب','تمام','بعض',
    'کئی','چند','تھوڑا','زیادہ','بہت','کم','اتنا','اتنی','جتنا','جتنی',
]
for w in DETS:
    if w not in POS_LEXICON:
        POS_LEXICON[w] = 'DET'

CONJS = [
    'اور','یا','لیکن','مگر','بلکہ','تاہم','چنانچہ','لہذا','کیونکہ','کہ',
    'جب','جہاں','جیسے','اگر','تو','پھر','تاکہ','حتی کہ','نہ','نہیں',
]
for w in CONJS:
    POS_LEXICON[w] = 'CONJ'

POSTS = [
    'میں','پر','سے','کو','کے','کی','کا','نے','تک','تلک',
    'ساتھ','بعد','پہلے','اندر','باہر','اوپر','نیچے','آگے','پیچھے','درمیان',
    'بارے','خلاف','حق','لیے','واسطے','علاوہ','سوا','بجائے','طرف','جانب',
]
for w in POSTS:
    if w not in POS_LEXICON:
        POS_LEXICON[w] = 'POST'

ADVS = [
    'بھی','صرف','تو','ہی','نہیں','نہ','کبھی','ہمیشہ','اکثر','کبھی کبھی',
    'آج','کل','پرسوں','ابھی','پھر','اب','پہلے','بعد','جلدی','دھیرے',
    'یہاں','وہاں','کہاں','ادھر','ادھر','اوپر','نیچے','دور','قریب','آگے',
    'ہاں','جی','نہیں','واقعی','یقیناً','شاید','ضرور','البتہ','بالکل','بالخصوص',
]
for w in ADVS:
    if w not in POS_LEXICON:
        POS_LEXICON[w] = 'ADV'

NUMS = [
    'ایک','دو','تین','چار','پانچ','چھ','سات','آٹھ','نو','دس',
    'گیارہ','بارہ','تیرہ','چودہ','پندرہ','سولہ','سترہ','اٹھارہ','انیس','بیس',
    'تیس','چالیس','پچاس','ساٹھ','ستر','اسی','نوے','سو','ہزار','لاکھ',
    'کروڑ','ارب','پہلا','دوسرا','تیسرا','نصف','ربع','دگنا','تگنا','چوگنا',
    '۱','۲','۳','۴','۵','۶','۷','۸','۹','۰',
]
for w in NUMS:
    POS_LEXICON[w] = 'NUM'

PUNCS = ['۔', '،', '؟', '!', ':', ';', '"', "'", '(', ')', '-', '—', '...', '…']
for w in PUNCS:
    POS_LEXICON[w] = 'PUNC'

print("Lexicon entries : {0}".format(len(POS_LEXICON)))
print("Entries per tag:")
tagCounts = Counter(POS_LEXICON.values())
for tag, cnt in sorted(tagCounts.items()):
    print("  {0:<6}: {1}".format(tag, cnt))

### Rule-Based POS Tagger

In [ ]:
# lexicon lookup first, then morphological suffix rules
def rule_pos_tag(tokens):
    """
    Rule-based POS tagger using lexicon + morphological rules.
    Returns list of (token, tag) pairs.
    """
    tagged = []
    for i, tok in enumerate(tokens):
        if tok in POS_LEXICON:
            tagged.append((tok, POS_LEXICON[tok]))
            continue

        if re.match(r"^[۔،؟!:;\"'()\-—…]+$", tok):
            tagged.append((tok, 'PUNC'))
            continue

        if re.match(r'^[۰-۹0-9][۰-۹0-9,٬.]*$', tok):
            tagged.append((tok, 'NUM'))
            continue

        if (tok.endswith('نا') or tok.endswith('نے') or tok.endswith('تا') or
            tok.endswith('تی') or tok.endswith('تے') or tok.endswith('ئی') or
            tok.endswith('ئے') or tok.endswith('ئیں') or tok.endswith('یں') or
            tok.endswith('یا') or tok.endswith('کر') or tok.endswith('کے')):
            tagged.append((tok, 'VERB'))
            continue

        if (tok.endswith('انہ') or tok.endswith('آور') or tok.endswith('مند') or
            tok.endswith('پذیر') or tok.endswith('ناک') or tok.endswith('گین') or
            tok.endswith('دار') or tok.endswith('ی') and len(tok) > 3):
            tagged.append((tok, 'ADJ'))
            continue

        if (tok.endswith('ستان') or tok.endswith('ہاں') or tok.endswith('گاہ') or
            tok.endswith('خانہ') or tok.endswith('کاری') or tok.endswith('گری') or
            tok.endswith('بازی') or tok.endswith('پرستی') or tok.endswith('واری')):
            tagged.append((tok, 'NOUN'))
            continue

        tagged.append((tok, 'NOUN'))

    return tagged

sampleSent = selected[0]['tokens']
print("Sample POS annotation:")
print("Tokens :", sampleSent)
print("Tagged  :", rule_pos_tag(sampleSent))

### 3.3 NER Annotation (BIO Scheme + Gazetteer)

In [ ]:
PERSONS = {
    'عمران خان','نواز شریف','شہباز شریف','آصف علی زرداری','بلاول بھٹو',
    'مریم نواز','پرویز مشرف','بے نظیر بھٹو','ذوالفقار علی بھٹو','ایوب خان',
    'یحیی خان','ضیا الحق','اسحاق ڈار','محسن نقوی','شفقت محمود',
    'فواد چوہدری','شیخ رشید','اعتزاز احسن','اسد عمر','پرویز الہی',
    'چوہدری نثار','جاوید لطیف','مصطفی کمال','الطاف حسین','فاروق ستار',
    'سراج الحق','مولانا فضل الرحمان','اختر مینگل','محمود خان اچکزئی','اسفند یار ولی',
    'جنرل باجوہ','جنرل عاصم منیر','جنرل راحیل','کیانی','ظفر الله',
    'عطا تارڑ','رانا ثنا الله','خواجہ آصف','احسن اقبال','چوہدری شجاعت',
    'ریحام خان','جمائمہ گولڈ اسمتھ','طاہر القادری','قادری','منظور پشتین',
    'ملالہ یوسف زئی','ڈاکٹر عبدالقدیر خان','ڈاکٹر ثمر','پرویز ہود بھائی','عرفان صدیقی',
    'ثاقب نثار','گلزار احمد','عمر عطا بندیال','فائز عیسی','منصور علی شاہ',
}

LOCATIONS = {
    'پاکستان','بھارت','انڈیا','چین','امریکہ','برطانیہ','ایران','افغانستان','سعودی عرب','متحدہ عرب امارات',
    'لاہور','کراچی','اسلام آباد','پشاور','کوئٹہ','ملتان','فیصل آباد','راولپنڈی','حیدرآباد','سکھر',
    'گجرانوالہ','سیالکوٹ','بہاول پور','مردان','ابوٹ آباد','مظفرآباد','گلگت','سکردو','چترال','دیر',
    'پنجاب','سندھ','بلوچستان','خیبر پختونخوا','گلگت بلتستان','آزاد کشمیر','فاٹا','اسلام آباد',
    'کشمیر','وزیرستان','سوات','مالاکنڈ','بنوں','ڈیرہ اسماعیل خان','ٹانک','لکی مروت',
    'واشنگٹن','نیویارک','لندن','بیجنگ','ماسکو','تہران','کابل','دہلی','ممبئی','ڈھاکہ',
    'دبئی','ریاض','انقرہ','اسرائیل','فلسطین','لبنان','شام','عراق','یمن','مصر',
    'یوکرین','روس','جرمنی','فرانس','اٹلی','سپین','کینیڈا','آسٹریلیا','جاپان','کوریا',
}

ORGANISATIONS = {
    'پاکستان تحریک انصاف','پاکستان مسلم لیگ','پاکستان پیپلز پارٹی','جمعیت علمائے اسلام','عوامی نیشنل پارٹی',
    'ایم کیو ایم','بی این پی','پختونخوا ملی عوامی پارٹی','جماعت اسلامی','تحریک لبیک',
    'پاک فوج','آئی ایس آئی','آئی ایس پی آر','ایف سی','رینجرز',
    'سپریم کورٹ','ہائی کورٹ','نیب','ایف آئی اے','ایف بی آر',
    'اقوام متحدہ','آئی ایم ایف','ورلڈ بینک','ایشیائی ترقیاتی بینک','سارک',
    'بی بی سی','جیو نیوز','آرٹی وائی','ڈان نیوز','سماء ٹی وی',
    'پی سی بی','فیفا','آئی سی سی','ایشیا کپ','ورلڈ کپ',
    'ناسا','پی آئی اے','ریلوے','واپڈا','پی ٹی سی ایل',
}

MISC_ENTITIES = {
    'کووڈ','کورونا','اومیکرون','ڈیلٹا','ایڈز','ملیریا','ٹائیفائڈ','ڈینگی',
    'آئین','منشور','بجٹ','سی پیک','گوادر پورٹ','خلیج','دریائے سندھ',
}

NER_SINGLE = {}
for name in PERSONS:
    parts = name.split()
    if len(parts) == 1:
        NER_SINGLE[parts[0]] = 'PER'
for loc in LOCATIONS:
    parts = loc.split()
    if len(parts) == 1:
        NER_SINGLE[parts[0]] = 'LOC'
for org in ORGANISATIONS:
    parts = org.split()
    if len(parts) == 1:
        NER_SINGLE[parts[0]] = 'ORG'
for misc in MISC_ENTITIES:
    parts = misc.split()
    if len(parts) == 1:
        NER_SINGLE[parts[0]] = 'MISC'

MULTI_GAZ = {}
for name in PERSONS:
    MULTI_GAZ[tuple(name.split())] = 'PER'
for loc in LOCATIONS:
    MULTI_GAZ[tuple(loc.split())] = 'LOC'
for org in ORGANISATIONS:
    MULTI_GAZ[tuple(org.split())] = 'ORG'
for misc in MISC_ENTITIES:
    MULTI_GAZ[tuple(misc.split())] = 'MISC'

print("Gazetteer persons : {0}".format(len(PERSONS)))
print("Gazetteer locations : {0}".format(len(LOCATIONS)))
print("Gazetteer organisations : {0}".format(len(ORGANISATIONS)))
print("Gazetteer MISC : {0}".format(len(MISC_ENTITIES)))
print("Total multi-token entries : {0}".format(len(MULTI_GAZ)))

In [ ]:
def nerTag(tokens):

    n = len(tokens)
    tags = ['O'] * n
    i = 0
    while i < n:
        matched = False
        for span_len in range(min(4, n - i), 0, -1):
            span = tuple(tokens[i:i + span_len])
            if span in MULTI_GAZ:
                etype = MULTI_GAZ[span]
                tags[i] = f'B-{etype}'
                for j in range(1, span_len):
                    tags[i + j] = f'I-{etype}'
                i += span_len
                matched = True
                break
        if not matched:
            tok = tokens[i]
            if tok in NER_SINGLE:
                tags[i] = f'B-{NER_SINGLE[tok]}'
            i += 1
    return list(zip(tokens, tags))

sampleToks = ['عمران', 'خان', 'نے', 'لاہور', 'میں', 'کہا']
print("NER sample:", nerTag(sampleToks))

### 3.4 Annotated 500 Sentences & Train/Val/Test Split

In [ ]:
annotated = []
for item in selected:
    toks = item['tokens']
    pos = rule_pos_tag(toks)
    ner = nerTag(toks)
    annotated.append({'aid': item['aid'],'topic' : item['topic'],'tokens': toks,'pos': [tag for _, tag in pos],'ner': [tag for _, tag in ner],})

from collections import defaultdict

byTopic = defaultdict(list)
for item in annotated:
    byTopic[item['topic']].append(item)

trainData, valData, testData = [], [], []
for topic, items in byTopic.items():
    random.shuffle(items)
    n = len(items)
    nTrain = int(0.70 * n)
    nVal = int(0.15 * n)
    trainData += items[:nTrain]
    valData += items[nTrain:nTrain + nVal]
    testData += items[nTrain + nVal:]

random.shuffle(trainData)
random.shuffle(valData)
random.shuffle(testData)

print("Train: {0} | Val: {1} | Test: {2}".format(len(trainData), len(valData), len(testData)))
print()

print("POS tag distribution (train):")
posTrainTags = [t for item in trainData for t in item['pos']]
for tag, cnt in sorted(Counter(posTrainTags).items()):
    print("  {0:<6}: {1}".format(tag, cnt))

print()
print("NER tag distribution (train):")
nerTrainTags = [t for item in trainData for t in item['ner']]
for tag, cnt in sorted(Counter(nerTrainTags).items()):
    print("  {0:<8}: {1}".format(tag, cnt))

In [ ]:
# write dataset to CoNLL format for external tools
def save_conll(data, filepath, task='pos'):
    """Save data in CoNLL format: TOKEN TAG per line, blank line between sentences."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            tags = item[task]
            for tok, tag in zip(item['tokens'], tags):
                f.write(f'{tok}\t{tag}\n')
            f.write('\n')

save_conll(trainData, 'data/pos_train.conll', 'pos')
save_conll(testData,  'data/pos_test.conll',  'pos')
save_conll(trainData, 'data/ner_train.conll', 'ner')
save_conll(testData,  'data/ner_test.conll',  'ner')

print("Saved: data/pos_train.conll, data/pos_test.conll")
print("Saved: data/ner_train.conll, data/ner_test.conll")

## 4. BiLSTM Sequence Labeler

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

POS_TAGS = ['NOUN','VERB','ADJ','ADV','PRON','DET','CONJ','POST','NUM','PUNC','UNK']
NER_TAGS = ['O','B-PER','I-PER','B-LOC','I-LOC','B-ORG','I-ORG','B-MISC','I-MISC']

pos2idx = {t: i for i, t in enumerate(POS_TAGS)}
ner2idx = {t: i for i, t in enumerate(NER_TAGS)}
idx2pos = {i: t for t, i in pos2idx.items()}
idx2ner = {i: t for t, i in ner2idx.items()}

PAD_IDX  = 0
PAD_TAG  = -1

print("POS tags : {0}".format(len(pos2idx)))
print("NER tags : {0}".format(len(ner2idx)))

### Sequence Dataset

In [ ]:
# pads token and tag sequences for batch training
class SeqLabelDataset(Dataset):
    """
    Dataset for POS/NER sequence labeling.
    Returns padded token-id sequences and tag-id sequences.
    """
    def __init__(self, data, word2idx, tag2idx, task='pos'):
        self.samples = []
        for item in data:
            token_ids = [word2idx.get(t, 0) for t in item['tokens']]
            tag_ids   = [tag2idx.get(item[task][i], tag2idx.get('UNK', tag2idx.get('O', 0))) for i in range(len(item['tokens']))]
            self.samples.append((token_ids, tag_ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    """Pad sequences in a batch; return (paddedTokens, paddedTags, lengths)."""
    tokenSeqs, tagSeqs = zip(*batch)
    lengths = torch.tensor([len(s) for s in tokenSeqs], dtype=torch.long)
    lengths, sort_idx = lengths.sort(descending=True)
    tokenSeqs = [tokenSeqs[i] for i in sort_idx]
    tagSeqs = [tagSeqs[i]   for i in sort_idx]

    max_len = int(lengths[0].item())
    paddedTokens = torch.zeros(len(batch), max_len, dtype=torch.long)
    paddedTags = torch.full((len(batch), max_len), PAD_TAG, dtype=torch.long)

    for i, (toks, tags) in enumerate(zip(tokenSeqs, tagSeqs)):
        L = len(toks)
        paddedTokens[i, :L] = torch.tensor(toks, dtype=torch.long)
        paddedTags[i, :L] = torch.tensor(tags, dtype=torch.long)

    return paddedTokens, paddedTags, lengths

posTrainDs = SeqLabelDataset(trainData, vocab, pos2idx, 'pos')
posValDs = SeqLabelDataset(valData, vocab, pos2idx, 'pos')
posTestDs = SeqLabelDataset(testData, vocab, pos2idx, 'pos')

nerTrainDs = SeqLabelDataset(trainData, vocab, ner2idx, 'ner')
nerValDs = SeqLabelDataset(valData, vocab, ner2idx, 'ner')
nerTestDs  = SeqLabelDataset(testData, vocab, ner2idx, 'ner')

BATCH_SIZE_SEQ = 32
posTrainLoader = DataLoader(posTrainDs, batch_size=BATCH_SIZE_SEQ, shuffle=True,  collate_fn=collate_fn)
posValLoader = DataLoader(posValDs, batch_size=BATCH_SIZE_SEQ, shuffle=False, collate_fn=collate_fn)
posTestLoader  = DataLoader(posTestDs, batch_size=BATCH_SIZE_SEQ, shuffle=False, collate_fn=collate_fn)
nerTrainLoader = DataLoader(nerTrainDs, batch_size=BATCH_SIZE_SEQ, shuffle=True,  collate_fn=collate_fn)
nerValLoader = DataLoader(nerValDs, batch_size=BATCH_SIZE_SEQ, shuffle=False, collate_fn=collate_fn)
nerTestLoader  = DataLoader(nerTestDs, batch_size=BATCH_SIZE_SEQ, shuffle=False, collate_fn=collate_fn)

print("POS — train: {0} | val: {1} | test: {2}".format(len(posTrainDs), len(posValDs), len(posTestDs)))
print("NER — train: {0} | val: {1} | test: {2}".format(len(nerTrainDs), len(nerValDs), len(nerTestDs)))

### CRF Layer (for NER)

In [ ]:
# linear-chain CRF with Viterbi decoding
class CRF(nn.Module):
    """
    Linear-chain CRF with learnable tag-transition matrix.
    Inference via Viterbi algorithm.
    """
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags   = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags) * 0.1)

    def _score_sentence(self, emissions, tags, mask):
        """Compute gold-sequence score for training (numerator of CRF loss)."""
        B, T, C = emissions.shape
        score = torch.zeros(B, device=emissions.device)
        score += emissions[torch.arange(B), 0, tags[:, 0]]
        for t in range(1, T):
            active = mask[:, t]
            trans  = self.transitions[tags[:, t-1], tags[:, t]]
            emit   = emissions[torch.arange(B), t, tags[:, t]]
            score  += (trans + emit) * active.float()
        return score

    def _forward_alg(self, emissions, mask):
        """Forward algorithm to compute log partition function Z."""
        B, T, C = emissions.shape
        alpha = emissions[:, 0, :]
        for t in range(1, T):
            active = mask[:, t].unsqueeze(1).float()
            alpha_expanded = alpha.unsqueeze(2)
            trans_expanded = self.transitions.unsqueeze(0)
            alpha_new = torch.logsumexp(alpha_expanded + trans_expanded, dim=1) + emissions[:, t, :]
            alpha = alpha_new * active + alpha * (1 - active)
        return torch.logsumexp(alpha, dim=1)

    def neg_log_likelihood(self, emissions, tags, mask):
        """CRF NLL loss = log Z - score(gold)."""
        forward_score = self._forward_alg(emissions, mask)
        gold_score    = self._score_sentence(emissions, tags, mask)
        return (forward_score - gold_score).mean()

    def viterbi_decode(self, emissions, mask):
        """Viterbi decoding; returns best tag sequence per sentence."""
        B, T, C = emissions.shape
        viterbi   = emissions[:, 0, :]
        backtrack = []

        for t in range(1, T):
            scores = viterbi.unsqueeze(2) + self.transitions.unsqueeze(0)
            best_scores, best_tags = scores.max(dim=1)
            active = mask[:, t].unsqueeze(1).float()
            viterbi = (best_scores + emissions[:, t, :]) * active + viterbi * (1 - active)
            backtrack.append(best_tags)

        best_last = viterbi.argmax(dim=1)
        paths = [best_last.unsqueeze(1)]
        for bt in reversed(backtrack):
            best_last = bt[torch.arange(B), best_last]
            paths.append(best_last.unsqueeze(1))
        paths.reverse()
        return torch.cat(paths, dim=1)

print("CRF layer defined.")

### BiLSTM Model

In [ ]:
# 2-layer BiLSTM with optional CRF output layer
class BiLSTMTagger(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags,
                 pretrained_emb=None, freeze_emb=True, use_crf=False):
        super().__init__()
        self.use_crf    = use_crf
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(
                torch.tensor(pretrained_emb, dtype=torch.float32))
        self.embedding.weight.requires_grad = not freeze_emb

        self.lstm = nn.LSTM(
            input_size    = embed_dim,
            hidden_size   = hidden_dim,
            num_layers    = 2,
            bidirectional = True,
            batch_first   = True,
            dropout        = 0.5,
        )
        self.dropout = nn.Dropout(p=0.5)

        self.fc = nn.Linear(hidden_dim * 2, num_tags)

        if use_crf:
            self.crf = CRF(num_tags)

    def forward(self, token_ids, lengths):
        """
        token_ids : (B, T)  padded token indices
        lengths   : (B,)    actual sequence lengths
        Returns emissions (B, T, num_tags).
        """
        emb  = self.dropout(self.embedding(token_ids))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=True)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        out    = self.dropout(out)
        emissions = self.fc(out)
        return emissions

    def loss(self, token_ids, lengths, tags):
        """Compute loss; mask out padding positions."""
        emissions = self.forward(token_ids, lengths)
        B, T, C   = emissions.shape
        mask = torch.zeros(B, T, dtype=torch.bool, device=token_ids.device)
        for i, l in enumerate(lengths):
            mask[i, :l] = True

        if self.use_crf:
            return self.crf.neg_log_likelihood(emissions, tags.clamp(min=0), mask)
        else:
            return nn.CrossEntropyLoss(ignore_index=PAD_TAG)(
                emissions.view(-1, C), tags.view(-1))

    def predict(self, token_ids, lengths):
        """Return predicted tag ids (unpadded per sentence)."""
        self.eval()
        with torch.no_grad():
            emissions = self.forward(token_ids, lengths)
            B, T, C   = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=token_ids.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True

            if self.use_crf:
                paths = self.crf.viterbi_decode(emissions, mask)
                preds = [paths[i, :lengths[i]].tolist() for i in range(B)]
            else:
                preds_padded = emissions.argmax(dim=-1)
                preds = [preds_padded[i, :lengths[i]].tolist() for i in range(B)]
        return preds

_emb = np.load('embeddings/embC3.npy')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

_model = BiLSTMTagger(vocab_size = len(vocab),embed_dim = 100,hidden_dim = 128,num_tags = len(pos2idx),pretrained_emb= _emb,freeze_emb = True,use_crf = False,).to(device)

_ids  = torch.randint(0, len(vocab), (4, 10)).to(device)
_lens = torch.tensor([10, 8, 6, 4], dtype=torch.long)
_out  = _model(_ids, _lens)
print("BiLSTM output shape : {0}  (expected 4×10×{1})".format(_out.shape, len(pos2idx)))
del _model, _ids, _lens, _out

### Training & Early Stopping

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import time

# token-level macro F1 on a data loader
def compute_f1(model, loader, device, idx2tag):
    """Token-level F1 on a data loader (ignoring padding)."""
    allTrue, allPred = [], []
    model.eval()
    with torch.no_grad():
        for token_ids, tags, lengths in loader:
            token_ids = token_ids.to(device)
            preds = model.predict(token_ids, lengths)
            for i, (pred, length) in enumerate(zip(preds, lengths)):
                gold = tags[i, :length].tolist()
                allTrue.extend(gold)
                allPred.extend(pred)
    labels = [v for v in idx2tag.keys() if v != PAD_TAG]
    f1 = f1_score(allTrue, allPred, labels=labels, average='macro', zero_division=0)
    return f1

# train with early stopping on validation F1
def train_bilstm(model, train_loader, val_loader, device,
                 n_epochs=20, lr=1e-3, weight_decay=1e-4,
                 patience=5, label='', idx2tag=None):
    """
    Train BiLSTM with early stopping on validation F1.
    Returns (trainLosses, valLosses, valF1s).
    """
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    bestValF1   = -1.0
    bestState    = None
    patienceCnt  = 0
    trainLosses, valLosses, valF1s = [], [], []

    print("Training: {0}".format(label))
    print("Epochs={0}  LR={1}  Patience={2}".format(n_epochs, lr, patience))

    for epoch in range(n_epochs):
        model.train()
        epochLoss = 0.0
        t0 = time.time()
        for token_ids, tags, lengths in train_loader:
            token_ids = token_ids.to(device)
            tags      = tags.to(device)
            optimizer.zero_grad()
            loss = model.loss(token_ids, lengths, tags)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epochLoss += loss.item()

        avgTrain = epochLoss / len(train_loader)
        trainLosses.append(avgTrain)

        model.eval()
        valLoss = 0.0
        with torch.no_grad():
            for token_ids, tags, lengths in val_loader:
                token_ids = token_ids.to(device)
                tags = tags.to(device)
                valLoss += model.loss(token_ids, lengths, tags).item()
        avgVal = valLoss / len(val_loader)
        valLosses.append(avgVal)

        valF1 = compute_f1(model, val_loader, device, idx2tag or idx2pos)
        valF1s.append(valF1)

        print("  Epoch {0:>2}/{1} | Train Loss: {2:.4f} | Val Loss: {3:.4f} | Val F1: {4:.4f} | {5:.1f}s".format(epoch+1, n_epochs, avgTrain, avgVal, valF1, time.time()-t0))

        if valF1 > bestValF1:
            bestValF1 = valF1
            bestState  = {k: v.clone() for k, v in model.state_dict().items()}
            patienceCnt = 0
        else:
            patienceCnt += 1
            if patienceCnt >= patience:
                print("  Early stopping at epoch {0}".format(epoch+1))
                break

    if bestState is not None:
        model.load_state_dict(bestState)
    print("  Best Val F1 : {0:.4f}".format(bestValF1))
    return trainLosses, valLosses, valF1s

def plot_curves(trainLosses, valLosses, valF1s, title):
    """Plot training/validation loss and F1 per epoch."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(trainLosses) + 1)
    ax1.plot(epochs, trainLosses, label='Train Loss')
    ax1.plot(epochs, valLosses,   label='Val Loss', linestyle='--')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, valF1s, label='Val F1', color='green')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
    ax2.set_title(f'{title} — Val F1'); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

print("Training utilities defined.")

### 4.1 POS Tagger Training: Frozen vs Fine-Tuned Embeddings

In [ ]:
embC3 = np.load('embeddings/embC3.npy')

posModelFrozen = BiLSTMTagger(
    vocab_size    = len(vocab),
    embed_dim     = 100,
    hidden_dim    = 128,
    num_tags      = len(pos2idx),
    pretrained_emb= embC3,
    freeze_emb    = True,
    use_crf       = False,
).to(device)

trainLF, valLF, valF1F = train_bilstm(
    posModelFrozen, posTrainLoader, posValLoader, device,
    n_epochs=20, lr=1e-3, weight_decay=1e-4, patience=5,
    label='POS — Frozen Embeddings', idx2tag=idx2pos)

plot_curves(trainLF, valLF, valF1F, 'POS (Frozen Embeddings)')
torch.save(posModelFrozen.state_dict(), 'models/bilstm_pos.pt')
print("Saved models/bilstm_pos.pt")

In [ ]:
posModelFt = BiLSTMTagger(
    vocab_size    = len(vocab),
    embed_dim     = 100,
    hidden_dim    = 128,
    num_tags      = len(pos2idx),
    pretrained_emb= embC3,
    freeze_emb    = False,
    use_crf       = False,
).to(device)

trainLFt, valLFt, valF1Ft = train_bilstm(
    posModelFt, posTrainLoader, posValLoader, device,
    n_epochs=20, lr=1e-3, weight_decay=1e-4, patience=5,
    label='POS — Fine-Tuned Embeddings', idx2tag=idx2pos)

plot_curves(trainLFt, valLFt, valF1Ft, 'POS (Fine-Tuned Embeddings)')

print("\nEmbedding mode comparison (Val F1):")
print("  Frozen     : {0:.4f}".format(max(valF1F)))
print("  Fine-tuned : {0:.4f}".format(max(valF1Ft)))

### 4.2 NER Tagger Training: With and Without CRF

In [ ]:
nerModelCrf = BiLSTMTagger(
    vocab_size    = len(vocab),
    embed_dim     = 100,
    hidden_dim    = 128,
    num_tags      = len(ner2idx),
    pretrained_emb= embC3,
    freeze_emb    = False,
    use_crf       = True,
).to(device)

trainLCrf, valLCrf, valF1Crf = train_bilstm(
    nerModelCrf, nerTrainLoader, nerValLoader, device,
    n_epochs=20, lr=1e-3, weight_decay=1e-4, patience=5,
    label='NER — BiLSTM + CRF', idx2tag=idx2ner)

plot_curves(trainLCrf, valLCrf, valF1Crf, 'NER (BiLSTM + CRF)')
torch.save(nerModelCrf.state_dict(), 'models/bilstm_ner.pt')
print("Saved models/bilstm_ner.pt")

In [ ]:
nerModelNocrf = BiLSTMTagger(
    vocab_size    = len(vocab),
    embed_dim     = 100,
    hidden_dim    = 128,
    num_tags      = len(ner2idx),
    pretrained_emb= embC3,
    freeze_emb    = False,
    use_crf       = False,
).to(device)

trainLNc, valLNc, valF1Nc = train_bilstm(
    nerModelNocrf, nerTrainLoader, nerValLoader, device,
    n_epochs=20, lr=1e-3, weight_decay=1e-4, patience=5,
    label='NER — BiLSTM (No CRF)', idx2tag=idx2ner)

plot_curves(trainLNc, valLNc, valF1Nc, 'NER (No CRF)')

print("\nCRF vs No-CRF comparison (Best Val F1):")
print("  With CRF    : {0:.4f}".format(max(valF1Crf)))
print("  Without CRF : {0:.4f}".format(max(valF1Nc)))

## 5. Evaluation

### 5.1 POS Tagging Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# collect flat prediction and gold lists from a loader
def get_all_preds(model, loader, device):
    """Collect all predictions and gold labels (flat lists, no padding)."""
    allTrue, allPred = [], []
    model.eval()
    with torch.no_grad():
        for token_ids, tags, lengths in loader:
            token_ids = token_ids.to(device)
            preds = model.predict(token_ids, lengths)
            for i, (pred, length) in enumerate(zip(preds, lengths)):
                gold = tags[i, :length].tolist()
                allTrue.extend(gold)
                allPred.extend(pred)
    return allTrue, allPred

posTrue, posPred = get_all_preds(posModelFt, posTestLoader, device)

posTrueTags = [idx2pos[t] for t in posTrue]
posPredTags = [idx2pos[p] for p in posPred]

acc = accuracy_score(posTrueTags, posPredTags)
print("POS Test Accuracy : {0:.4f}".format(acc))
print()
print("Classification Report:")
print(classification_report(posTrueTags, posPredTags,labels=POS_TAGS, zero_division=0))

In [ ]:
cm = confusion_matrix(posTrueTags, posPredTags, labels=POS_TAGS)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(POS_TAGS))); ax.set_xticklabels(POS_TAGS, rotation=45, ha='right')
ax.set_yticks(range(len(POS_TAGS))); ax.set_yticklabels(POS_TAGS)
ax.set_xlabel('Predicted Tag'); ax.set_ylabel('True Tag')
ax.set_title('POS Tagging — Confusion Matrix (Test Set)')
for i in range(len(POS_TAGS)):
    for j in range(len(POS_TAGS)):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=7)
plt.tight_layout(); plt.show()

errors = []
for i in range(len(POS_TAGS)):
    for j in range(len(POS_TAGS)):
        if i != j and cm[i, j] > 0:
            errors.append((cm[i, j], POS_TAGS[i], POS_TAGS[j]))
errors.sort(reverse=True)
print("Top-3 most confused tag pairs (true → predicted):")
for cnt, t, p in errors[:3]:
    print("  {0} → {1} : {2} errors".format(t, p, cnt))

In [ ]:
posTrueSeq, posPredSeq = [], []
testSents = []
posModelFt.eval()
with torch.no_grad():
    for token_ids, tags, lengths in posTestLoader:
        token_ids = token_ids.to(device)
        preds = posModelFt.predict(token_ids, lengths)
        for i, (pred, length) in enumerate(zip(preds, lengths)):
            posPredSeq.append(pred)
            posTrueSeq.append(tags[i, :length].tolist())

print("Example sentences for most confused tag pairs:\n")
topPairs = [(t, p) for _, t, p in errors[:3]]
shown = {pair: 0 for pair in topPairs}

for item in testData:
    toks = item['tokens']
    gold = [idx2pos.get(pos2idx.get(t, 0), 'UNK') for t in item['pos']]
    predTags = rule_pos_tag(toks)
    for (trueTag, predTag) in topPairs:
        if shown[(trueTag, predTag)] >= 2:
            continue
        for k, (tok, gtag) in enumerate(zip(toks, gold)):
            ptag = predTags[k][1]
            if gtag == trueTag and ptag == predTag:
                print("  Confused: {0} → {1}".format(trueTag, predTag))
                print("  Sentence: {0}".format(' '.join(toks)))
                print("  Token '{0}' tagged as {1} (should be {2})\n".format(tok, predTag, trueTag))
                shown[(trueTag, predTag)] += 1
                break

In [ ]:
posTrueFrz, posPredFrz = get_all_preds(posModelFrozen, posTestLoader, device)
accFrz = accuracy_score([idx2pos[t] for t in posTrueFrz],[idx2pos[p] for p in posPredFrz])
f1_frz = f1_score([idx2pos[t] for t in posTrueFrz],[idx2pos[p] for p in posPredFrz],labels=POS_TAGS, average='macro', zero_division=0)

accFt = accuracy_score(posTrueTags, posPredTags)
f1_ft = f1_score(posTrueTags, posPredTags,labels=POS_TAGS, average='macro', zero_division=0)

print("POS — Frozen vs Fine-Tuned Embedding Comparison")
print("{0:<15} {1:>10} {2:>10}".format('Mode', 'Accuracy', 'Macro-F1'))
print( )
print("{0:<15} {1:>10.4f} {2:>10.4f}".format('Frozen', accFrz, f1_frz))
print("{0:<15} {1:>10.4f} {2:>10.4f}".format('Fine-Tuned', accFt, f1_ft))

### 5.2 NER Evaluation

In [ ]:
# entity-level P/R/F1 using conlleval counting logic
def conlleval_score(true_seqs, pred_seqs, idx2tag):
    """
    Entity-level precision, recall, F1 per type.
    """
    def get_entities(seq):
        entities = []
        start, etype = None, None
        for i, tag in enumerate(seq):
            if tag.startswith('B-'):
                if start is not None:
                    entities.append((etype, start, i - 1))
                start, etype = i, tag[2:]
            elif tag.startswith('I-'):
                if start is None or etype != tag[2:]:
                    start, etype = i, tag[2:]
            else:
                if start is not None:
                    entities.append((etype, start, i - 1))
                    start, etype = None, None
        if start is not None:
            entities.append((etype, start, len(seq) - 1))
        return entities

    tp = Counter(); fp = Counter(); fn = Counter()
    for true_s, pred_s in zip(true_seqs, pred_seqs):
        true_ents = set(get_entities(true_s))
        pred_ents = set(get_entities(pred_s))
        for e in true_ents & pred_ents:
            tp[e[0]] += 1
        for e in pred_ents - true_ents:
            fp[e[0]] += 1
        for e in true_ents - pred_ents:
            fn[e[0]] += 1

    types = sorted(set(list(tp.keys()) + list(fp.keys()) + list(fn.keys())))
    print("{0:<8} {1:>8} {2:>8} {3:>8}".format('Type', 'Prec', 'Rec', 'F1'))
    print( )
    all_tp = all_fp = all_fn = 0
    for t in types:
        p  = tp[t] / (tp[t] + fp[t] + 1e-10)
        r  = tp[t] / (tp[t] + fn[t] + 1e-10)
        f  = 2*p*r / (p + r + 1e-10)
        print("{0:<8} {1:>8.4f} {2:>8.4f} {3:>8.4f}".format(t, p, r, f))
        all_tp += tp[t]; all_fp += fp[t]; all_fn += fn[t]
    p_all = all_tp / (all_tp + all_fp + 1e-10)
    r_all = all_tp / (all_tp + all_fn + 1e-10)
    f_all = 2*p_all*r_all / (p_all + r_all + 1e-10)
    print( )
    print("{0:<8} {1:>8.4f} {2:>8.4f} {3:>8.4f}".format('Overall', p_all, r_all, f_all))
    return f_all

# collect NER tag sequences (not flat) from a loader
def get_ner_seqs(model, loader, device, idx2tag):
    true_seqs, pred_seqs = [], []
    model.eval()
    with torch.no_grad():
        for token_ids, tags, lengths in loader:
            token_ids = token_ids.to(device)
            preds = model.predict(token_ids, lengths)
            for i, (pred, length) in enumerate(zip(preds, lengths)):
                gold = tags[i, :length].tolist()
                true_seqs.append([idx2tag[t] for t in gold])
                pred_seqs.append([idx2tag[p] for p in pred])
    return true_seqs, pred_seqs

print(" NER with CRF")
nerTrueCrf, nerPredCrf = get_ner_seqs(nerModelCrf, nerTestLoader, device, idx2ner)
f1_crf = conlleval_score(nerTrueCrf, nerPredCrf, idx2ner)

print("\nNER without CRF")
nerTrueNc, nerPredNc = get_ner_seqs(nerModelNocrf, nerTestLoader, device, idx2ner)
f1_nc = conlleval_score(nerTrueNc, nerPredNc, idx2ner)

print("\nCRF vs No-CRF Overall F1: {0:.4f} vs {1:.4f}".format(f1_crf, f1_nc))

In [ ]:
# print sample false positives and false negatives
def error_analysis(true_seqs, pred_seqs, testData, n=5):
    """Print FP and FN entity examples."""
    def get_entities_with_text(seq, tokens):
        entities = []
        start, etype = None, None
        for i, tag in enumerate(seq):
            if tag.startswith('B-'):
                if start is not None:
                    entities.append((etype, start, i-1, ' '.join(tokens[start:i])))
                start, etype = i, tag[2:]
            elif tag.startswith('I-'):
                pass
            else:
                if start is not None:
                    entities.append((etype, start, i-1, ' '.join(tokens[start:i])))
                    start, etype = None, None
        if start is not None:
            entities.append((etype, start, len(seq)-1, ' '.join(tokens[start:])))
        return set((e[0], e[3]) for e in entities)

    fp_list, fn_list = [], []
    for i, (true_s, pred_s) in enumerate(zip(true_seqs, pred_seqs)):
        toks = testData[i % len(testData)]['tokens']
        true_ents = get_entities_with_text(true_s, toks)
        pred_ents = get_entities_with_text(pred_s, toks)
        for e in pred_ents - true_ents:
            fp_list.append((e, ' '.join(toks)))
        for e in true_ents - pred_ents:
            fn_list.append((e, ' '.join(toks)))

    print("5 False Positives (predicted entity, not in gold)")
    for (etype, span), sent in fp_list[:n]:
        print("  [{0}] '{1}'  in: {2:<80}...".format(etype, span, sent[:80]))

    print("\n5 False Negatives (gold entity, missed by model)")
    for (etype, span), sent in fn_list[:n]:
        print("  [{0}] '{1}'  in: {2:80}...".format(etype, span, sent[:80]))

error_analysis(nerTrueCrf, nerPredCrf, testData)

### 5.3 Ablation Study (A1 – A4)

In [ ]:
# ablation A1: unidirectional LSTM
class UniLSTMTagger(nn.Module):
    """Unidirectional LSTM for ablation A1."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags,pretrained_emb=None, use_crf=False):
        super().__init__()
        self.use_crf = use_crf
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(torch.tensor(pretrained_emb, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,bidirectional=False, batch_first=True, dropout=0.5)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, num_tags)
        if use_crf:
            self.crf = CRF(num_tags)

    def forward(self, token_ids, lengths):
        emb = self.dropout(self.embedding(token_ids))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=True)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        return self.fc(self.dropout(out))

    def loss(self, token_ids, lengths, tags):
        emissions = self.forward(token_ids, lengths)
        B, T, C   = emissions.shape
        mask = torch.zeros(B, T, dtype=torch.bool, device=token_ids.device)
        for i, l in enumerate(lengths): mask[i, :l] = True
        if self.use_crf:
            return self.crf.neg_log_likelihood(emissions, tags.clamp(min=0), mask)
        return nn.CrossEntropyLoss(ignore_index=PAD_TAG)(emissions.view(-1, C), tags.view(-1))

    def predict(self, token_ids, lengths):
        self.eval()
        with torch.no_grad():
            emissions = self.forward(token_ids, lengths)
            B, T, C   = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=token_ids.device)
            for i, l in enumerate(lengths): mask[i, :l] = True
            if self.use_crf:
                paths = self.crf.viterbi_decode(emissions, mask)
                return [paths[i, :lengths[i]].tolist() for i in range(B)]
            preds = emissions.argmax(dim=-1)
            return [preds[i, :lengths[i]].tolist() for i in range(B)]

# ablation A2: BiLSTM without dropout
class BiLSTMNoDropout(nn.Module):
    """BiLSTM without dropout for ablation A2."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags, pretrained_emb=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(
                torch.tensor(pretrained_emb, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, token_ids, lengths):
        emb = self.embedding(token_ids)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=True)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        return self.fc(out)

    def loss(self, token_ids, lengths, tags):
        emissions = self.forward(token_ids, lengths)
        B, T, C = emissions.shape
        return nn.CrossEntropyLoss(ignore_index=PAD_TAG)(emissions.view(-1, C), tags.view(-1))

    def predict(self, token_ids, lengths):
        self.eval()
        with torch.no_grad():
            emissions = self.forward(token_ids, lengths)
            preds = emissions.argmax(dim=-1)
            return [preds[i, :lengths[i]].tolist() for i in range(len(lengths))]

print("Ablation model classes defined.")

In [ ]:
ablationResults = {}

a1Model = UniLSTMTagger(len(vocab), 100, 128, len(pos2idx),pretrained_emb=embC3).to(device)
tl, vl, vf = train_bilstm(a1Model, posTrainLoader, posValLoader, device,n_epochs=20, lr=1e-3, patience=5,label='A1 Uni-LSTM', idx2tag=idx2pos)
a1True, a1Pred = get_all_preds(a1Model, posTestLoader, device)
ablationResults['A1 (Uni-LSTM)'] = f1_score([idx2pos[t] for t in a1True], [idx2pos[p] for p in a1Pred],labels=POS_TAGS, average='macro', zero_division=0)

a2Model = BiLSTMNoDropout(len(vocab), 100, 128, len(pos2idx),pretrained_emb=embC3).to(device)
tl, vl, vf = train_bilstm(a2Model, posTrainLoader, posValLoader, device,n_epochs=20, lr=1e-3, patience=5,label='A2 No Dropout', idx2tag=idx2pos)
a2True, a2Pred = get_all_preds(a2Model, posTestLoader, device)
ablationResults['A2 (No Dropout)'] = f1_score([idx2pos[t] for t in a2True], [idx2pos[p] for p in a2Pred],labels=POS_TAGS, average='macro', zero_division=0)

a3Model = BiLSTMTagger(len(vocab), 100, 128, len(pos2idx),pretrained_emb=None, freeze_emb=False).to(device)
tl, vl, vf = train_bilstm(a3Model, posTrainLoader, posValLoader, device,n_epochs=20, lr=1e-3, patience=5,label='A3 Random Emb', idx2tag=idx2pos)
a3True, a3Pred = get_all_preds(a3Model, posTestLoader, device)
ablationResults['A3 (Random Emb)'] = f1_score([idx2pos[t] for t in a3True], [idx2pos[p] for p in a3Pred],labels=POS_TAGS, average='macro', zero_division=0)

a4Model = BiLSTMTagger(len(vocab), 100, 128, len(ner2idx),pretrained_emb=embC3, freeze_emb=False,use_crf=False).to(device)
tl, vl, vf = train_bilstm(a4Model, nerTrainLoader, nerValLoader, device,n_epochs=20, lr=1e-3, patience=5,label='A4 NER No CRF', idx2tag=idx2ner)
a4TrueSeqs, a4PredSeqs = get_ner_seqs(a4Model, nerTestLoader, device, idx2ner)

def flat_f1_ner(true_seqs, pred_seqs):
    flat_t = [t for s in true_seqs for t in s]
    flat_p = [p for s in pred_seqs for p in s]
    return f1_score(flat_t, flat_p, labels=NER_TAGS, average='macro', zero_division=0)

ablationResults['A4 (NER No CRF)'] = flat_f1_ner(a4TrueSeqs, a4PredSeqs)

baseF1 = f1_score(posTrueTags, posPredTags, labels=POS_TAGS,
                    average='macro', zero_division=0)
ablationResults['Baseline (BiLSTM+CRF)'] = baseF1

print("\n")
print("Ablation Study Results — Macro F1")
print( )
for name, score in ablationResults.items():
    print("  {0:<30} : {1:.4f}".format(name, score))

## Part 3) Transformer Encoder for Topic Classification

## 6. Dataset Preparation

In [ ]:
import os, re, json, random, math, time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

os.makedirs('models', exist_ok=True)

# map topic strings to integer class labels
CATEGORIES = ['Politics', 'Sports', 'Economy', 'International', 'Health_Society']
catToIdx = {c: i for i, c in enumerate(CATEGORIES)}
idxToCat = {i: c for c, i in catToIdx.items()}

print("Category → index mapping:")
for c, i in catToIdx.items():
    print("  {0}  {1}".format(i, c))
print()

catCounts = Counter(articleTopics[aid] for aid in doc_ids)
print("Article distribution across categories:")
for c in CATEGORIES:
    print("  {0}: {1}".format(c, catCounts.get(c, 0)))

In [ ]:
MAX_SEQ_LEN = 256
CLS_TOKEN_IDX = len(vocab)
PAD_TOKEN_IDX = 0

# truncate/pad article tokens to fixed length
def article_to_ids(aid, max_len=MAX_SEQ_LEN):
    tokens = tokenized_cleaned.get(aid, [])
    ids = [vocab.get(t, 0) for t in tokens[:max_len]]
    ids += [PAD_TOKEN_IDX] * (max_len - len(ids))
    return ids

allEntries = []
for aid in doc_ids:
    ids = article_to_ids(aid)
    label = catToIdx[articleTopics[aid]]
    allEntries.append({'aid': aid, 'ids': ids, 'label': label})

print("Total articles: {0}".format(len(allEntries)))
print("Sequence length: {0} tokens (padded/truncated)".format(MAX_SEQ_LEN))

In [ ]:
from collections import defaultdict

byCat = defaultdict(list)
for entry in allEntries:
    byCat[articleTopics[entry['aid']]].append(entry)

trainCls, valCls, testCls = [], [], []
for cat, items in byCat.items():
    random.shuffle(items)
    n = len(items)
    nTrain = int(0.70 * n)
    nVal   = int(0.15 * n)
    trainCls += items[:nTrain]
    valCls   += items[nTrain:nTrain + nVal]
    testCls  += items[nTrain + nVal:]

random.shuffle(trainCls)
random.shuffle(valCls)
random.shuffle(testCls)

print("Train: {0} | Val: {1} | Test: {2}".format(len(trainCls), len(valCls), len(testCls)))
print()
print("Class distribution (train):")
for c, i in catToIdx.items():
    cnt = sum(1 for e in trainCls if e['label'] == i)
    print("  {0}: {1}".format(c, cnt))
print()
print("Class distribution (val):")
for c, i in catToIdx.items():
    cnt = sum(1 for e in valCls if e['label'] == i)
    print("  {0}: {1}".format(c, cnt))
print()
print("Class distribution (test):")
for c, i in catToIdx.items():
    cnt = sum(1 for e in testCls if e['label'] == i)
    print("  {0}: {1}".format(c, cnt))

## 7. Transformer Encoder

### 7.1 Scaled Dot-Product Attention

In [ ]:
# Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V
class ScaledDotProductAttention(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):

        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights

_sdpa = ScaledDotProductAttention()
_q = torch.randn(2, 4, 8, 32)
_k = torch.randn(2, 4, 8, 32)
_v = torch.randn(2, 4, 8, 32)
_out, _attn = _sdpa(_q, _k, _v)
print("SDPA output shape : {0}".format(_out.shape))
print("SDPA attn shape : {0}".format(_attn.shape))
del _sdpa, _q, _k, _v, _out, _attn

### 7.2 Multi-Head Self-Attention

In [ ]:
# h=4 heads, project then split, shared output projection
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model=128, num_heads=4):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        self.d_model   = d_model

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.attn = ScaledDotProductAttention()

    def forward(self, x, mask=None):

        B, T, _ = x.shape

        Q = self.W_q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        attn_mask = None
        if mask is not None:
            attn_mask = mask.unsqueeze(1).unsqueeze(2)

        out, attn_weights = self.attn(Q, K, V, attn_mask)

        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.W_o(out)
        return out, attn_weights

_mha = MultiHeadSelfAttention(d_model=128, num_heads=4)
_x   = torch.randn(2, 10, 128)
_out, _aw = _mha(_x)
print("MHSA output shape : {0}".format(_out.shape))
print("MHSA attn_weights shape : {0}".format(_aw.shape))
del _mha, _x, _out, _aw

### 7.3 Position-wise Feed-Forward Network

In [ ]:
# FFN(x) = ReLU(xW1 + b1)W2 + b2, d_ff=512
class PositionwiseFFN(nn.Module):

    def __init__(self, d_model=128, d_ff=512, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

_ffn = PositionwiseFFN(d_model=128, d_ff=512)
_x = torch.randn(2, 10, 128)
print("FFN output shape : {0}".format(_ffn(_x).shape))
del _ffn, _x

### 7.4 Sinusoidal Positional Encoding

In [ ]:
# fixed sinusoidal PE stored as a non-learned buffer
class SinusoidalPositionalEncoding(nn.Module):

    def __init__(self, d_model=128, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) *
            (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """x : (B, T, d_model)"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

### 7.5 Pre-Layer Norm Encoder Block

In [ ]:
# pre-LN encoder block: MHSA + FFN with residuals
class EncoderBlock(nn.Module):

    def __init__(self, d_model=128, num_heads=4, d_ff=512, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mhsa  = MultiHeadSelfAttention(d_model, num_heads)
        self.ffn   = PositionwiseFFN(d_model, d_ff, dropout)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):

        residual = x
        x_norm   = self.norm1(x)
        attn_out, attn_weights = self.mhsa(x_norm, mask)
        x = residual + self.drop(attn_out)

        residual = x
        x = residual + self.drop(self.ffn(self.norm2(x)))

        return x, attn_weights

_blk = EncoderBlock(d_model=128, num_heads=4, d_ff=512)
_x   = torch.randn(2, 10, 128)
_out, _aw = _blk(_x)
print("EncoderBlock output shape : {0}".format(_out.shape))
print("Attn weights shape : {0}".format(_aw.shape))
del _blk, _x, _out, _aw

### 7.6 Full Transformer Encoder Classifier

In [ ]:
# 4-block transformer encoder with [CLS] classification head
class TransformerClassifier(nn.Module):

    def __init__(self, vocab_size, num_classes=5,d_model=128, num_heads=4, d_ff=512,num_layers=4, max_len=257, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        self.token_emb = nn.Embedding(vocab_size + 1, d_model, padding_idx=0)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len, dropout)

        self.blocks = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout)for _ in range(num_layers)])

        self.norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(nn.Linear(d_model, 64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64, num_classes),)

    def forward(self, token_ids, return_attn=False):

        B, T = token_ids.shape

        pad_mask_tokens = (token_ids == 0)
        cls_mask = torch.zeros(B, 1, dtype=torch.bool,device=token_ids.device)
        pad_mask = torch.cat([cls_mask, pad_mask_tokens], dim=1)

        x = self.token_emb(token_ids)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        x = self.pos_enc(x)

        attn_list = []
        for block in self.blocks:
            x, attn_w = block(x, pad_mask)
            attn_list.append(attn_w)

        x = self.norm(x)

        cls_out = x[:, 0, :]
        logits = self.classifier(cls_out)

        if return_attn:
            return logits, attn_list
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device : {0}".format(device))

_model = TransformerClassifier(vocab_size=len(vocab)).to(device)
_ids = torch.randint(0, len(vocab), (4, MAX_SEQ_LEN)).to(device)
_logits = _model(_ids)
print("Transformer output shape : {0}".format(_logits.shape))

total_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print("Trainable parameters : {0:,}".format(total_params))
del _model, _ids, _logits

### 7.7 Dataset & DataLoader

In [ ]:
# wraps article token-ID lists and labels for the DataLoader
class TopicDataset(Dataset):
    def __init__(self, entries):
        self.entries = entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        return (torch.tensor(e['ids'],   dtype=torch.long),
                torch.tensor(e['label'], dtype=torch.long))

BATCH_SIZE_CLS = 16

trainClsDs = TopicDataset(trainCls)
valClsDs = TopicDataset(valCls)
testClsDs = TopicDataset(testCls)

trainClsLoader = DataLoader(trainClsDs, batch_size=BATCH_SIZE_CLS, shuffle=True)
valClsLoader = DataLoader(valClsDs, batch_size=BATCH_SIZE_CLS, shuffle=False)
testClsLoader = DataLoader(testClsDs, batch_size=BATCH_SIZE_CLS, shuffle=False)

print("Train batches : {0}".format(len(trainClsLoader)))
print("Val batches : {0}".format(len(valClsLoader)))
print("Test batches : {0}".format(len(testClsLoader)))

### 7.8 Cosine LR Schedule with Warmup

In [ ]:
# linear warmup then cosine decay
def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps):

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / max(1, warmup_steps)
        progress = float(current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print("Cosine LR schedule with warmup defined.")

### 7.9 Training Loop (20 Epochs, AdamW)

In [ ]:
# AdamW with cosine LR warmup, returns loss/acc history
def train_transformer(model, train_loader, val_loader, device, n_epochs=20, lr=5e-4, weight_decay=0.01, warmup_steps=50):

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = n_epochs * len(train_loader)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    trainLosses, valLosses = [], []
    trainAccs,   valAccs   = [], []

    print("Training Transformer Encoder")
    print("Epochs={0} | LR={1} | WD={2} | Warmup={3}".format(n_epochs, lr, weight_decay, warmup_steps))

    for epoch in range(n_epochs):
        model.train()
        ep_loss, ep_correct, ep_total = 0.0, 0, 0
        t0 = time.time()
        for token_ids, labels in train_loader:
            token_ids = token_ids.to(device)
            labels    = labels.to(device)
            optimizer.zero_grad()
            logits = model(token_ids)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            ep_loss    += loss.item()
            ep_correct += (logits.argmax(dim=-1) == labels).sum().item()
            ep_total   += labels.size(0)

        trainLosses.append(ep_loss / len(train_loader))
        trainAccs.append(ep_correct / ep_total)

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for token_ids, labels in val_loader:
                token_ids = token_ids.to(device)
                labels    = labels.to(device)
                logits    = model(token_ids)
                v_loss   += criterion(logits, labels).item()
                v_correct += (logits.argmax(dim=-1) == labels).sum().item()
                v_total   += labels.size(0)

        valLosses.append(v_loss / len(val_loader))
        valAccs.append(v_correct / v_total)

        print("  Epoch {0:>2}/{1} | Train Loss: {2:.4f}  Acc: {3:.4f} | Val Loss: {4:.4f}  Acc: {5:.4f} | {6:.1f}s".format(epoch+1, n_epochs, trainLosses[-1], trainAccs[-1], valLosses[-1], valAccs[-1], time.time()-t0))

    return trainLosses, valLosses, trainAccs, valAccs

transformerModel = TransformerClassifier(vocab_size  = len(vocab),num_classes = 5,d_model = 128,num_heads = 4,d_ff = 512,num_layers = 4,max_len = MAX_SEQ_LEN + 1,  dropout = 0.1,).to(device)

trainLossesT, valLossesT, trainAccsT, valAccsT = train_transformer(
    transformerModel, trainClsLoader, valClsLoader, device,
    n_epochs=20, lr=5e-4, weight_decay=0.01, warmup_steps=50)

torch.save(transformerModel.state_dict(), 'models/transformer_cls.pt')
print("Saved models/transformer_cls.pt")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, len(trainLossesT) + 1)

ax1.plot(epochs, trainLossesT, label='Train Loss')
ax1.plot(epochs, valLossesT,   label='Val Loss', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Transformer — Training & Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, trainAccsT, label='Train Accuracy')
ax2.plot(epochs, valAccsT,   label='Val Accuracy', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Transformer — Training & Validation Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/transformer_training_curves.png', dpi=150)
plt.show()
print("Saved models/transformer_training_curves.png")

## 8. Evaluation

### 8.1 Test Accuracy, Macro-F1, and Confusion Matrix

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# collect true and predicted class labels from a loader
def evaluate_classifier(model, loader, device):
    """Return flat lists of true and predicted labels."""
    allTrue, allPred = [], []
    model.eval()
    with torch.no_grad():
        for token_ids, labels in loader:
            token_ids = token_ids.to(device)
            logits    = model(token_ids)
            preds     = logits.argmax(dim=-1).cpu().tolist()
            allPred.extend(preds)
            allTrue.extend(labels.tolist())
    return allTrue, allPred

trueLabels, predLabels = evaluate_classifier(transformerModel, testClsLoader, device)

testAcc = accuracy_score(trueLabels, predLabels)
testF1  = f1_score(trueLabels, predLabels, average='macro', zero_division=0)

print("Test Accuracy : {0:.4f}".format(testAcc))
print("Test Macro-F1 : {0:.4f}".format(testF1))
print()
print("Per-class Classification Report:")
targetNames = [idxToCat[i] for i in range(len(CATEGORIES))]
print(classification_report(trueLabels, predLabels,target_names=targetNames, zero_division=0))

In [ ]:
cm = confusion_matrix(trueLabels, predLabels)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
tickLabels = [c.replace('_', '\n') for c in CATEGORIES]
ax.set_xticks(range(5)); ax.set_xticklabels(tickLabels, fontsize=9)
ax.set_yticks(range(5)); ax.set_yticklabels(tickLabels, fontsize=9)
ax.set_xlabel('Predicted Category', fontsize=11)
ax.set_ylabel('True Category',      fontsize=11)
ax.set_title('Transformer Encoder — 5×5 Confusion Matrix (Test Set)', fontsize=12)
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=11,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig('models/transformer_confusion_matrix.png', dpi=150)
plt.show()
print("Saved models/transformer_confusion_matrix.png")

### 8.2 Attention Heatmaps (3 Articles, ≥2 Heads of Final Encoder Layer)

In [ ]:
# visualise attention weights for 2 heads of the final layer
def plot_attention_heatmap(token_ids, attn_weights_last, article_tokens,article_label, pred_label, article_idx):
    displayLen = min(21, len(article_tokens) + 1)
    labelsX = ['[CLS]'] + article_tokens[:displayLen - 1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"Article {article_idx} | True: {article_label} | Pred: {pred_label}\n" f"Final Encoder Layer — Head 1 (left) and Head 2 (right)",fontsize=11)

    for head_idx, ax in enumerate(axes):
        attn = attn_weights_last[head_idx, :displayLen, :displayLen].cpu().numpy()
        im = ax.imshow(attn, cmap='viridis', aspect='auto', vmin=0)
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_xticks(range(displayLen))
        ax.set_xticklabels(labelsX, rotation=90, fontsize=7)
        ax.set_yticks(range(displayLen))
        ax.set_yticklabels(labelsX, fontsize=7)
        ax.set_title(f'Head {head_idx + 1}', fontsize=10)
        ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

    plt.tight_layout()
    plt.savefig(f'models/attention_heatmap_article{article_idx}.png', dpi=150)
    plt.show()
    print("Saved models/attention_heatmap_article{0}.png".format(article_idx))

transformerModel.eval()
correctExamples = []
with torch.no_grad():
    for entry in testCls:
        if len(correctExamples) >= 3:
            break
        ids_t  = torch.tensor(entry['ids'], dtype=torch.long).unsqueeze(0).to(device)
        logits, attn_list = transformerModel(ids_t, return_attn=True)
        pred   = logits.argmax(dim=-1).item()
        if pred == entry['label']:
            correctExamples.append({
                'entry' : entry,
                'pred' : pred,
                'attn_last' : attn_list[-1].squeeze(0),
            })

print("Found {0} correctly classified articles for heatmaps.".format(len(correctExamples)))

for i, ex in enumerate(correctExamples):
    aid    = ex['entry']['aid']
    tokens = tokenized_cleaned.get(aid, [])[:20]
    true_l = idxToCat[ex['entry']['label']]
    pred_l = idxToCat[ex['pred']]
    plot_attention_heatmap(
        token_ids = ex['entry']['ids'],
        attn_weights_last = ex['attn_last'],
        article_tokens = tokens,
        article_label = true_l,
        pred_label = pred_l,
        article_idx = i + 1,
    )

### 8.3 BiLSTM vs. Transformer Comparison

In [ ]:
# lightweight BiLSTM for topic classification comparison
class BiLSTMClassifier(nn.Module):
    """Simple 2-layer BiLSTM for topic classification (for comparison)."""
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128,
                 num_classes=5, pretrained_emb=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(
                torch.tensor(pretrained_emb, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            bidirectional=True, batch_first=True, dropout=0.5)
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, token_ids):
        lengths = (token_ids != 0).sum(dim=1).clamp(min=1).cpu()
        emb     = self.dropout(self.embedding(token_ids))
        from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
        lengths_sorted, sort_idx = lengths.sort(descending=True)
        emb_sorted = emb[sort_idx]
        packed  = pack_padded_sequence(emb_sorted, lengths_sorted,
                                       batch_first=True, enforce_sorted=True)
        out, (h, _) = self.lstm(packed)
        h_fwd = h[-2]
        h_bwd = h[-1]
        h_cat = torch.cat([h_fwd, h_bwd], dim=1)
        _, unsort_idx = sort_idx.sort()
        h_cat = h_cat[unsort_idx]
        return self.fc(self.dropout(h_cat))

emb_c3_arr = np.load('embeddings/embC3.npy')

bilstmClsModel = BiLSTMClassifier(
    vocab_size    = len(vocab),
    embed_dim     = 100,
    hidden_dim    = 128,
    num_classes   = 5,
    pretrained_emb= emb_c3_arr,
).to(device)

criterionB  = nn.CrossEntropyLoss()
optimizerB  = optim.Adam(bilstmClsModel.parameters(), lr=1e-3)

trainLossesB, valLossesB = [], []
trainAccsB,   valAccsB   = [], []

print("Training BiLSTM Classifier for comparison...")
for epoch in range(20):
    bilstmClsModel.train()
    ep_loss, ep_corr, ep_tot = 0.0, 0, 0
    t0 = time.time()
    for token_ids, labels in trainClsLoader:
        token_ids = token_ids.to(device); labels = labels.to(device)
        optimizerB.zero_grad()
        logits = bilstmClsModel(token_ids)
        loss   = criterionB(logits, labels)
        loss.backward()
        optimizerB.step()
        ep_loss += loss.item()
        ep_corr += (logits.argmax(-1) == labels).sum().item()
        ep_tot  += labels.size(0)
    trainLossesB.append(ep_loss / len(trainClsLoader))
    trainAccsB.append(ep_corr / ep_tot)

    bilstmClsModel.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for token_ids, labels in valClsLoader:
            token_ids = token_ids.to(device); labels = labels.to(device)
            logits = bilstmClsModel(token_ids)
            v_loss += criterionB(logits, labels).item()
            v_corr += (logits.argmax(-1) == labels).sum().item()
            v_tot  += labels.size(0)
    valLossesB.append(v_loss / len(valClsLoader))
    valAccsB.append(v_corr / v_tot)
    print("  Epoch {0:>2}/20 | Train Loss: {1:.4f}  Acc: {2:.4f} | Val Loss: {3:.4f}  Acc: {4:.4f} | {5:.1f}s".format(epoch+1, trainLossesB[-1], trainAccsB[-1], valLossesB[-1], valAccsB[-1], time.time()-t0))

In [ ]:
bilstmTrue, bilstmPred = evaluate_classifier(bilstmClsModel, testClsLoader, device)
bilstmAcc = accuracy_score(bilstmTrue, bilstmPred)
bilstmF1  = f1_score(bilstmTrue, bilstmPred, average='macro', zero_division=0)

print()
print("BiLSTM vs. Transformer — Topic Classification Summary")
print()
print("{0:<25} {1:>10} {2:>12}".format('Metric', 'BiLSTM', 'Transformer'))
print()
print("{0:<25} {1:>10.4f} {2:>12.4f}".format('Test Accuracy', bilstmAcc, testAcc))
print("{0:<25} {1:>10.4f} {2:>12.4f}".format('Test Macro-F1', bilstmF1, testF1))
print("{0:<25} {1:>10} {2:>12}".format('Epochs trained', '20', '20'))
print("{0:<25} {1:>10.4f} {2:>12.4f}".format('Best Val Acc', max(valAccsB), max(valAccsT)))
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, 21)

axes[0].plot(epochs, trainLossesB, label='BiLSTM Train', color='steelblue')
axes[0].plot(epochs, valLossesB,   label='BiLSTM Val',   color='steelblue', linestyle='--')
axes[0].plot(epochs, trainLossesT, label='Transformer Train', color='darkorange')
axes[0].plot(epochs, valLossesT,   label='Transformer Val',   color='darkorange', linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('BiLSTM vs. Transformer — Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, trainAccsB, label='BiLSTM Train', color='steelblue')
axes[1].plot(epochs, valAccsB,   label='BiLSTM Val',   color='steelblue', linestyle='--')
axes[1].plot(epochs, trainAccsT, label='Transformer Train', color='darkorange')
axes[1].plot(epochs, valAccsT,   label='Transformer Val',   color='darkorange', linestyle='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('BiLSTM vs. Transformer — Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/bilstm_vs_transformer.png', dpi=150)
plt.show()
print("Saved models/bilstm_vs_transformer.png")